# Step 1:  Load the student table and the teacher’s proxies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%run /content/drive/MyDrive/m5_project/notebooks/features.ipynb      # student table + hierarchy + scorer

import lightgbm as lgb, gc

# free memory: tables this notebook does not need (skips any that don't exist)
for name in ['df', 'price_feats', 'teacher', 'test', 'p']:
    globals().pop(name, None)
gc.collect()

# teacher proxies (saved in teacher.ipynb)
OUT   = DRIVE / 'outputs'
saved = pd.read_parquet(OUT / 'teacher_proxies.parquet')
saved.columns = saved.columns.astype(int)                            # '1914' -> 1914
proxies = {name: saved.loc[name].reindex(sales_by_level[name].index) for name in TEACHER_LEVELS}

print({name: table.shape for name, table in proxies.items()})
print('student:', student.shape)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6096944, 8)
days: 847 -> 1941
L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series
(6098, 1095)
L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days
Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
--------

# Step 2: Student features, forecast windows and clusters

In [ ]:
# clues the student uses (same for every trial -> fair comparison)
ID_COLS    = ['item_id', 'dept_id', 'cat_id', 'store_id']
S_FEATURES = ([f'lag_{l}' for l in LAGS]                        # past sales, same weekday
              + [f'roll_mean_{w}' for w in WINDOWS]             # usual level
              + ['sell_price', 'price_vs_usual']                # price clues
              + ['wday', 'month', 'day_of_month', 'event_type', 'snap_CA']   # calendar
              + ID_COLS)                                        # which product / dept / store

for col in ID_COLS:
    student[col] = student[col].astype('category')

# the two forecast windows
VAL_DAYS  = list(range(VAL_START, TEST_START))
TEST_DAYS = list(range(TEST_START, TEST_START + HORIZON))

CLUSTER_SETS = {'store': 'store_id', 'dept': 'dept_id'}

for set_name, col in CLUSTER_SETS.items():
    sizes = student[col].value_counts().sort_index()
    print(f'{set_name} clusters ({len(sizes)}):', sizes.to_dict())

print('features:', len(S_FEATURES))

store clusters (2): {'CA_1': 2946957, 'CA_2': 2759631}
dept clusters (7): {'FOODS_1': 429335, 'FOODS_2': 595096, 'FOODS_3': 1562719, 'HOBBIES_1': 820141, 'HOBBIES_2': 286209, 'HOUSEHOLD_1': 1011906, 'HOUSEHOLD_2': 1001182}
features: 20


# Step 3: One trial = one model per cluster, two jobs

In [ ]:
JOBS = {   # name: (last training day, days to forecast)
    'val':  (TRAIN_END,      VAL_DAYS),     # Job A -> graded by TCV
    'test': (TEST_START - 1, TEST_DAYS),    # Job B -> graded by H-Pro (proxies) and Gold (real test)
}

def to_grid(rows, preds, days):
    """Long predictions -> bottom grid (6,098 series x 28 days); not-on-sale days = 0."""
    grid = rows[['series_id', 'd']].assign(pred=preds).pivot(index='series_id', columns='d', values='pred')
    return grid.reindex(index=bottom.index, columns=days).fillna(0)

def run_trial(params, set_name, seed):
    """Train one model per cluster for both jobs; return {'val': grid, 'test': grid}."""
    col = CLUSTER_SETS[set_name]
    grids = {}
    for job, (last_day, days) in JOBS.items():
        parts = []
        for cluster in student[col].cat.categories:
            in_cluster = student[col] == cluster
            train_rows = student[in_cluster & (student.d <= last_day)]
            pred_rows  = student[in_cluster & student.d.isin(days)]
            model = lgb.LGBMRegressor(objective='tweedie', random_state=seed, n_jobs=-1, verbose=-1, **params)
            model.fit(train_rows[S_FEATURES], train_rows['sales'])
            parts.append(to_grid(pred_rows, model.predict(pred_rows[S_FEATURES]), days))
        grids[job] = sum(parts)          # clusters don't overlap, so adding = joining
    return grids


_g = run_trial(dict(n_estimators=10), 'store', seed=0)
print({job: g.shape for job, g in _g.items()})

{'val': (6098, 28), 'test': (6098, 28)}


# Step 4: Random search over student settings (with checkpoints)
- 12 random LightGBM settings (seed 0). Each trial is trained in every store cluster and every dept cluster,
for both jobs (validation and test)

In [ ]:
S_SPACE = {
    'learning_rate':          [0.03, 0.05, 0.1, 0.2],
    'num_leaves':             [31, 63, 127, 255],
    'min_child_samples':      [20, 50, 100, 200],
    'n_estimators':           [200, 400, 700],
    'colsample_bytree':       [0.6, 0.8, 1.0],
    'tweedie_variance_power': [1.1, 1.3, 1.5],
}
N_TRIALS = 12
SEED     = 0
rng = np.random.default_rng(SEED)
TRIAL_PARAMS = [{k: rng.choice(v).item() for k, v in S_SPACE.items()} for _ in range(N_TRIALS)]

CKPT = OUT / f'student_trials_seed{SEED}'
CKPT.mkdir(exist_ok=True)

import time
RUN_TRIALS = range(0, N_TRIALS)          # teammates can split: range(0, 6) and range(6, 12)
for t in RUN_TRIALS:
    params = TRIAL_PARAMS[t]
    for set_name in CLUSTER_SETS:
        f_val, f_test = CKPT / f'{set_name}_{t:02d}_val.parquet', CKPT / f'{set_name}_{t:02d}_test.parquet'
        if f_val.exists() and f_test.exists():
            print(f'trial {t:02d} {set_name:<5} already done, skipping'); continue
        start = time.time()
        grids = run_trial(params, set_name, SEED)
        for job, f in [('val', f_val), ('test', f_test)]:
            g = grids[job].copy(); g.columns = g.columns.astype(str); g.to_parquet(f)
        print(f'trial {t:02d} {set_name:<5} done in {(time.time() - start) / 60:.1f} min')

trial 00 store already done, skipping
trial 00 dept  already done, skipping
trial 01 store already done, skipping
trial 01 dept  already done, skipping
trial 02 store already done, skipping
trial 02 dept  already done, skipping
trial 03 store already done, skipping
trial 03 dept  already done, skipping
trial 04 store already done, skipping
trial 04 dept  already done, skipping
trial 05 store already done, skipping
trial 05 dept  already done, skipping
trial 06 store already done, skipping
trial 06 dept  already done, skipping
trial 07 store already done, skipping
trial 07 dept  already done, skipping
trial 08 store already done, skipping
trial 08 dept  already done, skipping
trial 09 store already done, skipping
trial 09 dept  already done, skipping
trial 10 store already done, skipping
trial 10 dept  already done, skipping
trial 11 store already done, skipping
trial 11 dept  already done, skipping


# Step 5: Grade every trial with every method
- TCV-Lowest, TCV-Hier, H-Pro-Top, H-Pro-Avg, Gold

In [ ]:
def load_grid(set_name, t, job):
    g = pd.read_parquet(CKPT / f'{set_name}_{t:02d}_{job}.parquet')
    g.columns = g.columns.astype(int)
    return g

def scaled_sq_errors(grid, targets, last_day, level_names):
    """For each level: (target - aggregated forecast)^2 / scale, per series per day, plus weights."""
    scales, weights = get_scales(last_day), get_weights(last_day)
    out = {}
    for name in level_names:
        pred = roll_up(grid, LEVELS[name])
        err  = (targets[name][grid.columns] - pred) ** 2
        out[name] = (err.div(scales[name], axis=0), weights[name])
    return out

def horizon_score(errs):
    """Paper RMSSE over the whole 28-day horizon, weighted, then averaged over the given levels."""
    return np.mean([(w * np.sqrt(e.mean(axis=1))).sum() for e, w in errs.values()])

def daily_score(errs):
    """Same, but separately for each of the 28 days (used by -PO)."""
    return sum((np.sqrt(e).mul(w, axis=0)).sum() for e, w in errs.values()) / len(errs)

ALL_LEVELS = list(LEVELS)
METHODS = {   # name: (forecast job graded, target tables, levels)
    'TCV-Lowest': ('val',  sales_by_level, ['L8_item_store']),
    'TCV-Hier':   ('val',  sales_by_level, ALL_LEVELS),
    'H-Pro-Top':  ('test', proxies,        ['L1_total']),
    'H-Pro-Avg':  ('test', proxies,        TEACHER_LEVELS),
    'Gold':       ('test', sales_by_level, ALL_LEVELS),
}
LAST_DAY = {'val': TRAIN_END, 'test': TEST_START - 1}

DONE = [t for t in range(N_TRIALS)
        if all((CKPT / f'{s}_{t:02d}_{j}.parquet').exists() for s in CLUSTER_SETS for j in JOBS)]
print('finished trials:', DONE)
grids = {(s, t, j): load_grid(s, t, j) for s in CLUSTER_SETS for t in DONE for j in JOBS}

rows, daily = [], {}
for set_name in CLUSTER_SETS:
    for t in DONE:
        row = {'set': set_name, 'trial': t}
        for method, (job, targets, levels) in METHODS.items():
            errs = scaled_sq_errors(grids[(set_name, t, job)], targets, LAST_DAY[job], levels)
            row[method] = horizon_score(errs)
            daily[(set_name, t, method)] = daily_score(errs).to_numpy()
        rows.append(row)
trial_scores = pd.DataFrame(rows)
trial_scores.to_csv(OUT / f'student_trial_scores_seed{SEED}.csv', index=False)
print(trial_scores.round(3).to_string(index=False))

# sanity: our Gold/TCV-Hier numbers must equal the hierarchy score() function
g = grids[('store', DONE[0], 'test')]; assert abs(score(g, TEST_START-1)[1] - trial_scores.iloc[0]['Gold']) < 1e-9
g = grids[('store', DONE[0], 'val')];  assert abs(score(g, TRAIN_END)[1] - trial_scores.iloc[0]['TCV-Hier']) < 1e-9

print(trial_scores.set_index(['set', 'trial']).groupby(level='set').idxmin())
print('scorer check ok')

finished trials: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  set  trial  TCV-Lowest  TCV-Hier  H-Pro-Top  H-Pro-Avg  Gold
store      0       0.854     0.663      0.463      0.486 0.841
store      1       0.849     0.640      0.390      0.426 0.772
store      2       0.861     0.702      0.543      0.543 0.913
store      3       0.871     0.698      0.517      0.513 0.903
store      4       0.860     0.705      0.482      0.488 0.857
store      5       0.854     0.661      0.473      0.489 0.858
store      6       0.847     0.615      0.358      0.402 0.750
store      7       0.848     0.631      0.377      0.417 0.766
store      8       0.857     0.696      0.489      0.497 0.870
store      9       0.882     0.777      0.609      0.589 0.973
store     10       0.866     0.653      0.459      0.465 0.862
store     11       0.858     0.665      0.438      0.451 0.842
 dept      0       0.857     0.632      0.455      0.465 0.867
 dept      1       0.842     0.638      0.422      0.438 0.794

# Step 6: Select trials: standard and per-offset (-PO)

- Standard: each method takes the trial with its best grade over the 28-day horizon.
- PO: the best trial for each forecast day, concatenated

In [ ]:
forecasts = {}
for set_name in CLUSTER_SETS:
    sc = trial_scores[trial_scores.set == set_name].set_index('trial')
    for method in METHODS:                                        # standard: best over the whole month
        best_t = sc[method].idxmin()
        forecasts[(set_name, method)] = grids[(set_name, best_t, 'test')]
    for method in ['TCV-Lowest', 'TCV-Hier', 'H-Pro-Top', 'H-Pro-Avg']:   # -PO: best per day
        per_day = np.vstack([daily[(set_name, t, method)] for t in DONE])
        best_per_day = [DONE[i] for i in per_day.argmin(axis=0)]
        forecasts[(set_name, method + '-PO')] = pd.DataFrame(
            {day: grids[(set_name, best_per_day[h], 'test')][day] for h, day in enumerate(TEST_DAYS)})
        print(f'{set_name:<5} {method + "-PO":<14} trials used per day:', best_per_day)
    print(set_name, 'chosen trial per method:', {m: int(sc[m].idxmin()) for m in METHODS})
print(len(forecasts), 'forecasts ready')

store TCV-Lowest-PO  trials used per day: [10, 11, 11, 0, 6, 6, 6, 7, 10, 7, 4, 6, 6, 6, 10, 5, 5, 5, 6, 11, 6, 0, 7, 7, 5, 0, 0, 6]
store TCV-Hier-PO    trials used per day: [1, 1, 6, 0, 5, 6, 6, 6, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 5, 6, 1, 1, 1, 6, 6, 6]
store H-Pro-Top-PO   trials used per day: [6, 6, 6, 6, 6, 6, 7, 6, 6, 6, 6, 6, 6, 1, 6, 6, 7, 1, 6, 1, 6, 6, 1, 1, 1, 11, 6, 6]
store H-Pro-Avg-PO   trials used per day: [6, 6, 6, 6, 6, 6, 7, 6, 6, 6, 6, 6, 6, 4, 6, 6, 6, 1, 6, 1, 6, 6, 1, 1, 1, 11, 6, 6]
store chosen trial per method: {'TCV-Lowest': 6, 'TCV-Hier': 6, 'H-Pro-Top': 6, 'H-Pro-Avg': 6, 'Gold': 6}
dept  TCV-Lowest-PO  trials used per day: [11, 8, 11, 6, 7, 1, 7, 7, 7, 1, 1, 6, 6, 6, 1, 2, 6, 1, 1, 1, 1, 6, 1, 1, 1, 6, 1, 7]
dept  TCV-Hier-PO    trials used per day: [1, 6, 6, 0, 6, 6, 6, 0, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 7, 6, 6, 6, 6, 6, 6]
dept  H-Pro-Top-PO   trials used per day: [6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 7, 7, 6, 6, 7, 6, 6, 7, 6, 6, 6, 6, 6, 6, 

# Step 7:The store + dept average, ensembles, and the test score

In [ ]:
method_names = list(dict.fromkeys(m for _, m in forecasts))
final = {m: (forecasts[('store', m)] + forecasts[('dept', m)]) / 2 for m in method_names}   # store + dept average

final['Ens (1)'] = (final['H-Pro-Avg'] + final['H-Pro-Top']) / 2
final['Ens (2)'] = (final['H-Pro-Avg-PO'] + final['H-Pro-Top-PO']) / 2
final['Ens (3)'] = (final['Ens (1)'] + final['Ens (2)']) / 2

results = {}
for m, grid in final.items():                                   # grade on the REAL test month
    levels, r_h = score(grid, TEST_START - 1)
    results[m] = {'R_H': r_h, **levels.to_dict()}
results = pd.DataFrame(results).T

tcv = ['TCV-Lowest', 'TCV-Lowest-PO', 'TCV-Hier', 'TCV-Hier-PO']
best_tcv = results.loc[tcv, 'R_H'].idxmin()                     # best TCV baseline (as the paper does for ensemble 4)
final['Ens (4)'] = (final['Ens (3)'] + final[best_tcv]) / 2
levels, r_h = score(final['Ens (4)'], TEST_START - 1)
results.loc['Ens (4)'] = {'R_H': r_h, **levels.to_dict()}

results.round(3).to_csv(OUT / f'student_results_seed{SEED}.csv')
print(results.round(3))
print('best TCV baseline used in Ens (4):', best_tcv)

                 R_H  L1_total  L2_store  L3_cat  L4_dept  L5_store_cat  \
TCV-Lowest     0.745     0.703     0.716   0.690    0.688         0.724   
TCV-Hier       0.745     0.703     0.716   0.690    0.688         0.724   
H-Pro-Top      0.745     0.703     0.716   0.690    0.688         0.724   
H-Pro-Avg      0.745     0.703     0.716   0.690    0.688         0.724   
Gold           0.745     0.703     0.716   0.690    0.688         0.724   
TCV-Lowest-PO  0.782     0.757     0.772   0.738    0.731         0.770   
TCV-Hier-PO    0.754     0.717     0.729   0.703    0.699         0.735   
H-Pro-Top-PO   0.748     0.706     0.720   0.694    0.690         0.728   
H-Pro-Avg-PO   0.750     0.712     0.724   0.699    0.694         0.730   
Ens (1)        0.745     0.703     0.716   0.690    0.688         0.724   
Ens (2)        0.748     0.708     0.721   0.695    0.691         0.728   
Ens (3)        0.746     0.705     0.718   0.692    0.689         0.726   
Ens (4)        0.746     

In [9]:
for m in ['H-Pro-Top', 'H-Pro-Avg', 'TCV-Hier', 'Gold']:
    g = final[m].copy(); g.columns = g.columns.astype(str)
    g.to_parquet(OUT / f'final_forecast_{m}_seed{SEED}.parquet')